### This notebook shows examples of doing normalization and compute deviations and probabilities

#skip this cell.
#This was for library development purposes only.

import sys
from pathlib import Path

src_path = Path("src")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [2]:
import PyVisualFields
print(PyVisualFields.__file__)
print(PyVisualFields.__version__)

E:\Partners HealthCare Dropbox\Mohammad Eslami\GithubRepos\PyVisualField\src\PyVisualFields\__init__.py
2.0.8


### getnv(): 
get current normative setting/environment (from visualFields package)

In [3]:
from PyVisualFields import visualFields

######## Get the current normalization information
currentNV = visualFields.getnv()

$name
[1] "24-2"

$perimetry
[1] "static automated perimetry"

$strategy
[1] "SITA Standard"

$size
[1] "Size III"


### Example 
Investigate, compute the missing blocks, and global indices. </br>
Always use canonicalize_vf_df() to make the data format compatible

In [4]:
# Sometimes we may have a partial dataset. For example, UWHVF has sensitivity, TD, and PD values.

import pandas as pd
from PyVisualFields.utils import canonicalize_vf_df, canonicalize_vf_row
from PyVisualFields.utils import compute_missing_blocks
from PyVisualFields.utils import print_vf_summary, investigate_vf_df

url = "https://raw.githubusercontent.com/uw-biomedical-ml/uwhvf/master/CSV/VF_Data.csv"
df_UWHVF = pd.read_csv(url)
df_UWHVF.head()
df_VFs_py = canonicalize_vf_df(df_UWHVF)


### if the URL was not available
#from PyVisualFields import visualFields
#df_VFs_py = canonicalize_vf_df(visualFields.data_vfpwgRetest24d2())

#### now you can investigate the available information in the dataframe
print_vf_summary(df_VFs_py) # this function will print

info = investigate_vf_df(df_VFs_py)# this function will extract the information only


=== VF Data Summary ===
Rows: 28943
Columns: 184

Pointwise blocks:
  ✓ sensitivity (54 locations)
  ✓ total_deviation (54 locations)
  ✓ pattern_deviation (54 locations)
  ✗ total_deviation_probability (0 locations)
  ✗ pattern_deviation_probability (0 locations)

Global indices:
  ✓ md
  ✗ mdprob
  ✓ psd
  ✗ psdprob
  ✗ ght
  ✗ vfi
  ✓ msens
  ✗ ssens
  ✗ tmd
  ✗ tsd
  ✗ pmd
  ✓ gh

Metadata:
  ✓ patientid
  ✓ eyeid
  ✗ date
  ✓ age
  ✗ yearsfollowed
  ✗ duration


In [5]:
""" use the following function to fill the missed blocks """

from PyVisualFields.utils import compute_missing_blocks
from PyVisualFields.utils import vf_blocks, missing_blocks

print('---> blocks',vf_blocks(df_VFs_py))
print('---> missed blocks', missing_blocks(df_VFs_py))

print(df_VFs_py.shape)
df_VFs = compute_missing_blocks(df_VFs_py)

print('---> blocks',vf_blocks(df_VFs))
print('---> missed blocks', missing_blocks(df_VFs))
print(df_VFs.shape)

---> blocks {'sens': True, 'td': True, 'pd': True, 'tdp': False, 'pdp': False}
---> missed blocks ['tdp', 'pdp']
(28943, 184)
---> blocks {'sens': True, 'td': True, 'pd': True, 'tdp': True, 'pdp': True}
---> missed blocks []
(28943, 292)


In [6]:

""" in the case that you don't have missing blocks, but missing values, 
# e.g., some rows/patients of your data don't have TDP values 
# You can compute the values step by step, 
# step 1: create a subset of data that block is missing
# ...
# Pay attention to the order of functions below and inputs """

from PyVisualFields import visualFields

df_VFs_py = canonicalize_vf_df(visualFields.data_vfpwgRetest24d2())
df_td = visualFields.gettd(df_VFs_py) # get TD values
df_tdp = visualFields.gettdp(df_td) # get TD probability values
df_pd = visualFields.getpd(df_td) # get PD values
df_pdp = visualFields.getpdp(df_pd) # get PD probability values
# gh = visualFields.getgh(df_td) # get the general height
df_pdp.head()


,patientid,eyeid,date,time,age,type,fpr,fnr,fl,duration,...,pdp45,pdp46,pdp47,pdp48,pdp49,pdp50,pdp51,pdp52,pdp53,pdp54
0,1,OD,2008-08-13,00:00:00,53,pwg,0,0.0,0.00,00:00:00,...,0.95,0.95,0.95,0.95,0.950,0.98,0.95,0.95,0.95,0.99
1,1,OD,2008-08-20,00:00:00,53,pwg,0,0.0,0.06,00:00:00,...,0.95,0.95,0.95,0.95,0.950,0.95,0.95,0.95,0.95,0.95
2,1,OD,2008-08-27,00:00:00,53,pwg,0,0.0,0.00,00:00:00,...,0.95,0.95,0.95,0.95,0.005,0.95,0.95,0.95,0.95,0.95
3,1,OD,2008-09-03,00:00:00,53,pwg,0,0.0,0.06,00:00:00,...,0.95,0.95,0.95,0.05,0.020,0.95,0.95,0.95,0.95,0.95
4,1,OD,2008-09-10,00:00:00,53,pwg,0,0.0,0.13,00:00:00,...,0.95,0.95,0.95,0.05,0.020,0.95,0.95,0.95,0.95,0.95


In [7]:
""" in addition to missing blocks, we can also calculate global indices,
e.g. vfi, msens (MS), ssens, tmd, tsd, pmd, psd, gh,
getgl()
Notice that, in our new implementation, it will calculate only missed indexes. """

from PyVisualFields.utils import canonicalize_vf_df, canonicalize_vf_row
from PyVisualFields.utils import vf_blocks, missing_blocks
from PyVisualFields.utils import compute_missing_blocks
from PyVisualFields.utils import print_vf_summary, investigate_vf_df

df_VFs_py = canonicalize_vf_df(visualFields.data_vfpwgRetest24d2())

df_VFs = compute_missing_blocks(df_VFs_py)

# get global indices, e.g. psd, vfi, MS, gh, msens (MS), tmd (i.e. MD):
df_gi = visualFields.getgl(df_VFs) 
df_gi.head()

############# combine them later if you need:
#combined_df = pd.concat([df_VFs, df_gi], axis=1)
#final_df = combined_df.loc[:, ~combined_df.columns.duplicated()] 


==> py_getgl: Already available global indices: []
==> py_getgl: missing global indices to compute: ['msens', 'ssens', 'tmd', 'tsd', 'pmd', 'psd', 'gh', 'vfi']


,patientid,eyeid,date,time,age,type,fpr,fnr,fl,duration,msens,ssens,tmd,tsd,pmd,psd,gh,vfi
0,1,OD,2008-08-13,00:00:00,53,pwg,0,0.0,0.00,00:00:00,24.288462,6.800624,-6.110470,6.614332,-4.203546,6.644596,-1.988565,88.558160
1,1,OD,2008-08-20,00:00:00,53,pwg,0,0.0,0.06,00:00:00,26.500000,6.708204,-4.033858,6.833541,-4.121287,6.881649,0.011435,91.016661
2,1,OD,2008-08-27,00:00:00,53,pwg,0,0.0,0.00,00:00:00,24.615385,4.516436,-5.949572,4.303623,-3.599052,4.293826,-2.378612,89.417149
3,1,OD,2008-09-03,00:00:00,53,pwg,0,0.0,0.06,00:00:00,25.269231,4.182669,-5.250323,4.037272,-3.212375,4.052251,-2.067210,92.791128
4,1,OD,2008-09-10,00:00:00,53,pwg,0,0.0,0.13,00:00:00,25.596154,3.471224,-4.920857,3.146895,-3.063885,3.129952,-1.865947,93.502668


In [8]:
""" calculate the p-values of the globial indices """

df_gi_withProbs = visualFields.getglp(df_gi)
df_gi_withProbs.head()

==> py_getglp: available globals: ['msens', 'ssens', 'tmd', 'tsd', 'pmd', 'psd', 'gh', 'vfi']
==> py_getglp: missing globals: []
==> py_getglp: missing global probability indices: ['msensprob', 'tmdprob', 'pmdprob', 'ghprob', 'vfiprob', 'ssensprob', 'tsdprob', 'psdprob']


,patientid,eyeid,date,time,age,type,fpr,fnr,fl,duration,...,gh,vfi,msensprob,tmdprob,pmdprob,ghprob,vfiprob,ssensprob,tsdprob,psdprob
0,1,OD,2008-08-13,00:00:00,53,pwg,0,0.0,0.00,00:00:00,...,-1.988565,88.558160,0.005,0.005,0.005,0.005,0.005,0.005,0.005,0.005
1,1,OD,2008-08-20,00:00:00,53,pwg,0,0.0,0.06,00:00:00,...,0.011435,91.016661,0.020,0.005,0.005,0.950,0.005,0.005,0.005,0.005
2,1,OD,2008-08-27,00:00:00,53,pwg,0,0.0,0.00,00:00:00,...,-2.378612,89.417149,0.005,0.005,0.005,0.005,0.005,0.005,0.005,0.005
3,1,OD,2008-09-03,00:00:00,53,pwg,0,0.0,0.06,00:00:00,...,-2.067210,92.791128,0.005,0.005,0.005,0.005,0.005,0.010,0.005,0.005
4,1,OD,2008-09-10,00:00:00,53,pwg,0,0.0,0.13,00:00:00,...,-1.865947,93.502668,0.005,0.005,0.005,0.005,0.005,0.050,0.005,0.005


### ======= If you are interested in changing the default setting ============
of deviation and normalization analysis. You can also define a new one.

### Get the all predefined normalization settings (from visualFields package)

In [9]:
from PyVisualFields import visualFields

NormValues = visualFields.normvals()
NormInfo = visualFields.get_info_normvals()

''' 
==> comment: > pw: pointwise, classic: smooth
==> comment: > default is: sunyiu_24d2
'''

==> comment: > pw: pointwise, classic: smooth
==> comment: > default is: sunyiu_24d2

==> SettingName:  sunyiu_24d2_pw
  name: 24-2
  perimetry: static automated perimetry
  strategy: SITA Standard
  size: Size III

==> SettingName:  sunyiu_24d2
  name: 24-2
  perimetry: static automated perimetry
  strategy: SITA Standard
  size: Size III

==> SettingName:  sunyiu_24d2_pw_cps
  name: 24-2
  perimetry: static automated perimetry
  strategy: SITA Standard
  size: Size III

==> SettingName:  sunyiu_24d2_cps
  name: 24-2
  perimetry: static automated perimetry
  strategy: SITA Standard
  size: Size III

==> SettingName:  iowa_PC26_pw
  name: Iowa pointwise NVs for PC26
  perimetry: Static automated perimetry
  strategy: ZEST
  size: Size V

==> SettingName:  iowa_PC26_pw_cps
  name: Iowa pointwise NVs for PC26 with a CPS
  perimetry: Static automated perimetry
  strategy: ZEST
  size: Size V

==> SettingName:  iowa_Peri_pw
  name: Iowa pointwise NVs for Peri
  perimetry: Static automated 

' \n==> comment: > pw: pointwise, classic: smooth\n==> comment: > default is: sunyiu_24d2\n'

### set normalization values based on predefined ones (from visualFields package)

In [10]:
from PyVisualFields import visualFields

# we are going to change the setting from classic SUNY-IU to Iowa pointwise 
print('===> current NV setting:', )
currentNV = visualFields.getnv() 
target_predeifned_nv_name = 'iowa_Peri_pw'
visualFields.setnv(target_predeifned_nv_name)
print('===> new current NV setting:', )
currentNV = visualFields.getnv() # check it is set correctly:

===> current NV setting:
$name
[1] "24-2"

$perimetry
[1] "static automated perimetry"

$strategy
[1] "SITA Standard"

$size
[1] "Size III"
===> new current NV setting:
$name
[1] "Iowa pointwise NVs for Peri"

$perimetry
[1] "Static automated perimetry"

$strategy
[1] "ZEST"

$size
[1] "Size V"


### caculate new normalization values from a population and set it to be used:

In [11]:
from PyVisualFields import visualFields
import pickle

######## caculate new nv values and set it to be used:
df_VFs_py = visualFields.data_vfctrSunyiu24d2()
newNV_r, newNV_py = visualFields.nvgenerate(df_VFs_py, method = "pointwise",
                             name = "our_NV",
                             perimetry = "something",
                             strategy = "something",
                             size = "tmp")
visualFields.setnv(newNV_r)
print('===> current NV setting: ')
currentNV = visualFields.getnv() # check it is set correctly:

#''' notcie: this normalization will not be saved.
#We need to set for each session
#so we need to save and load them seperately, e.g.: '''
newNV_dict = { "newNV_r": newNV_r, "newNV_py": newNV_py }
pickle.dump( newNV_dict, open( "our_NV.pkl", "wb" ) )

loaded_dict = pickle.load( open( "our_NV.pkl", "rb" ) )
newNV_r = loaded_dict['newNV_r']
newNV_py = loaded_dict['newNV_py']
print('===> current NV setting: ')
visualFields.setnv(newNV_r) # set it to be used
currentNV = visualFields.getnv() # check it is set correctly:


===> current NV setting: 
$name
[1] "our_NV"

$perimetry
[1] "something"

$strategy
[1] "something"

$size
[1] "tmp"
===> current NV setting: 
$name
[1] "our_NV"

$perimetry
[1] "something"

$strategy
[1] "something"

$size
[1] "tmp"


In [12]:
#### if we like to reset to default and change the normalization values to the defalut of package: sunyiu_24d2
visualFields.setdefaults()
currentNV = visualFields.getnv() # check it is set correctly:

==> default normalization setting is set: sunyiu_24d2
$name
[1] "24-2"

$perimetry
[1] "static automated perimetry"

$strategy
[1] "SITA Standard"

$size
[1] "Size III"
